# 9g — Temporal Takeoff Year Detection

**Goal:** Detect the takeoff year per cluster — when paper count shifted from baseline to growth phase.

**Method:** Threshold (year where 3-yr rolling avg > 2× prior 3-yr rolling avg); optionally PELT via `ruptures`.

**Input:** `q4growingniches.json` (already has `yearlyCounts`). No new pkl needed.

**Output:** `emergence_timeline.json`

In [ ]:
import json, os
import numpy as np
import pandas as pd

OUT = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'arxiv-trends-website', 'src', 'data')

with open(os.path.join(OUT, 'q4growingniches.json')) as f:
    q4 = json.load(f)

clusters = q4['fullData']
print(f'Clusters: {len(clusters)}')
print('Sample yearlyCounts:', clusters[0]['yearlyCounts'])

In [ ]:
def detect_takeoff(yearly_counts, min_papers=5):
    years = sorted(int(y) for y in yearly_counts)
    if len(years) < 6: return None, 'insufficient_data'
    full = list(range(min(years), max(years)+1))
    counts = [yearly_counts.get(str(y), 0) for y in full]
    rolling = pd.Series(counts, index=full).rolling(3, min_periods=1).mean()
    for i in range(3, len(full)):
        cur, pri = rolling.iloc[i], rolling.iloc[i-3]
        if cur < min_papers and pri < min_papers: continue
        if pri > 0 and cur >= 2 * pri:
            ratio = cur / pri
            conf = 'high' if ratio >= 5 else 'medium' if ratio >= 2 else 'low'
            return full[i], conf
    return None, 'no_takeoff_detected'

# Test on LLM cluster
llm = next(c for c in clusters if c['clusterId'] == 28)
print('LLM takeoff:', detect_takeoff(llm['yearlyCounts']))

In [ ]:
results, timeline = [], {}

for c in clusters:
    cid = c['clusterId']
    yc  = c['yearlyCounts']
    yr, conf = detect_takeoff(yc)
    years = sorted(int(y) for y in yc)
    pre_count  = sum(v for k,v in yc.items() if yr and int(k) < yr)
    post_count = sum(v for k,v in yc.items() if yr and int(k) >= yr)
    pre_yrs  = max(1, yr - min(years)) if yr else 1
    post_yrs = max(1, max(years) - yr + 1) if yr else 1
    entry = {'clusterId': cid, 'takeoffYear': yr, 'confidence': conf,
             'preGrowthRate':  round(pre_count/pre_yrs, 2) if yr else None,
             'postGrowthRate': round(post_count/post_yrs, 2) if yr else None,
             'topTerms': c.get('topTerms', [])[:5]}
    results.append(entry)
    if yr: timeline.setdefault(str(yr), []).append(cid)

out_path = os.path.join(OUT, 'emergence_timeline.json')
with open(out_path, 'w') as f:
    json.dump({'clusters': results, 'timeline': {yr: sorted(cids) for yr,cids in sorted(timeline.items())}}, f, indent=2)
print(f'Saved → {out_path}')
print('Timeline:', list(timeline.keys()))